In [0]:
import pandas as pd

df = pd.read_csv(
    "/Volumes/workspace/default/dropout/data.csv",
    sep=";",
)


In [0]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [0]:
df.shape

(4424, 37)

In [0]:
import re

def clean_column(col):
    col = col.strip()                      # remove leading/trailing spaces
    col = col.replace("\t", "")            # remove tabs
    col = col.lower()                      # lowercase
    col = col.replace("'", "")             # remove apostrophes
    col = re.sub(r"[()]", "", col)         # remove parentheses
    col = col.replace("/", "_")            # replace slash
    col = re.sub(r"\s+", "_", col)         # replace spaces with underscore
    col = re.sub(r"[^a-z0-9_]", "", col)   # remove anything else invalid
    return col

df.columns = [clean_column(c) for c in df.columns]

df.columns

Index(['marital_status', 'application_mode', 'application_order', 'course',
       'daytime_evening_attendance', 'previous_qualification',
       'previous_qualification_grade', 'nacionality', 'mothers_qualification',
       'fathers_qualification', 'mothers_occupation', 'fathers_occupation',
       'admission_grade', 'displaced', 'educational_special_needs', 'debtor',
       'tuition_fees_up_to_date', 'gender', 'scholarship_holder',
       'age_at_enrollment', 'international',
       'curricular_units_1st_sem_credited',
       'curricular_units_1st_sem_enrolled',
       'curricular_units_1st_sem_evaluations',
       'curricular_units_1st_sem_approved', 'curricular_units_1st_sem_grade',
       'curricular_units_1st_sem_without_evaluations',
       'curricular_units_2nd_sem_credited',
       'curricular_units_2nd_sem_enrolled',
       'curricular_units_2nd_sem_evaluations',
       'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade',
       'curricular_units_2nd_sem_wit

In [0]:
spark_df = spark.createDataFrame(df)

In [0]:
spark_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.student_dropout_raw"
)

In [0]:
spark.sql("SELECT * FROM workspace.default.student_dropout_raw LIMIT 5").show()

+--------------+----------------+-----------------+------+--------------------------+----------------------+----------------------------+-----------+---------------------+---------------------+------------------+------------------+---------------+---------+-------------------------+------+-----------------------+------+------------------+-----------------+-------------+---------------------------------+---------------------------------+------------------------------------+---------------------------------+------------------------------+--------------------------------------------+---------------------------------+---------------------------------+------------------------------------+---------------------------------+------------------------------+--------------------------------------------+-----------------+--------------+-----+--------+
|marital_status|application_mode|application_order|course|daytime_evening_attendance|previous_qualification|previous_qualification_grade|nacionality|

In [0]:
spark.sql("""
SELECT target, COUNT(*) as count
FROM workspace.default.student_dropout_raw
GROUP BY target
ORDER BY count DESC
""").show()

+--------+-----+
|  target|count|
+--------+-----+
|Graduate| 2209|
| Dropout| 1421|
|Enrolled|  794|
+--------+-----+



In [0]:
spark.table("workspace.default.student_dropout_raw").printSchema()

root
 |-- marital_status: long (nullable = true)
 |-- application_mode: long (nullable = true)
 |-- application_order: long (nullable = true)
 |-- course: long (nullable = true)
 |-- daytime_evening_attendance: long (nullable = true)
 |-- previous_qualification: long (nullable = true)
 |-- previous_qualification_grade: double (nullable = true)
 |-- nacionality: long (nullable = true)
 |-- mothers_qualification: long (nullable = true)
 |-- fathers_qualification: long (nullable = true)
 |-- mothers_occupation: long (nullable = true)
 |-- fathers_occupation: long (nullable = true)
 |-- admission_grade: double (nullable = true)
 |-- displaced: long (nullable = true)
 |-- educational_special_needs: long (nullable = true)
 |-- debtor: long (nullable = true)
 |-- tuition_fees_up_to_date: long (nullable = true)
 |-- gender: long (nullable = true)
 |-- scholarship_holder: long (nullable = true)
 |-- age_at_enrollment: long (nullable = true)
 |-- international: long (nullable = true)
 |-- curric